In [1]:
def main(datasource="bigalpha_2026_stock_bar1m", start_date=None, end_date=None):
    import pandas as pd
    import dai

    table_name = datasource or "bigalpha_2026_stock_bar1m"
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    query_start = (start_ts - pd.Timedelta(days=45)).strftime("%Y-%m-%d 00:00:00")
    query_end = end_ts.strftime("%Y-%m-%d 23:59:59")

    sql = f"""
    WITH daily_ticket AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            CASE
                WHEN COUNT(*) >= 30 AND SUM(deal_number) > 0
                THEN SUM(amount) / (SUM(deal_number) + 1e-12)
                ELSE NULL
            END AS average_ticket_amount_daily
        FROM {table_name}
        GROUP BY date::DATE, instrument
    ),
    rolling_ticket AS (
        SELECT
            date,
            instrument,
            CASE
                WHEN COUNT(average_ticket_amount_daily) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
                ) >= 12
                THEN AVG(average_ticket_amount_daily) OVER (
                    PARTITION BY instrument
                    ORDER BY date
                    ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
                )
                ELSE NULL
            END AS ticket_amount_20d
        FROM daily_ticket
    ),
    ranked AS (
        SELECT
            date,
            instrument,
            percent_rank() OVER (
                PARTITION BY date
                ORDER BY ticket_amount_20d
            ) AS ticket_amount_cross_section_position_20d
        FROM rolling_ticket
        WHERE ticket_amount_20d IS NOT NULL
    )
    SELECT
        date,
        instrument,
        -ticket_amount_cross_section_position_20d AS factor
    FROM ranked
    ORDER BY date, instrument
    """

    df = dai.query(
        sql,
        filters={"date": [query_start, query_end]},
        compression=True,
    ).df()
    df["date"] = pd.to_datetime(df["date"])
    df = df[(df["date"] >= start_ts.normalize()) & (df["date"] <= end_ts.normalize())]
    return df[["date", "instrument", "factor"]]
